In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('../DATA/Featured_Data/gurgaon_properties_post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 36,0.82,3.0,2.0,New Property,850.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,New Property,1226.0,1.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,New Property,1000.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,Relatively New,1615.0,1.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,Relatively New,582.0,0.0,0.0,High


In [554]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished

In [594]:
# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession

In [4]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,High


In [6]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\SAHIL\AppData\Local\Temp\ipykernel_13976\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,Low
1,0,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,Low
2,0,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,Low
3,0,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,High
4,0,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,High


In [8]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

C:\Users\SAHIL\AppData\Local\Temp\ipykernel_13976\2576796079.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


In [9]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,0
1,0,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,0
2,0,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,0
3,0,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,2
4,0,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,2


In [10]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [11]:
new_df.head()

,property_type,price,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,...,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna road,agePossession_old,agePossession_under construction
0,0,0.82,3.0,2.0,850.0,0.0,0.0,0,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,0.95,2.0,2.0,1226.0,1.0,0.0,0,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,0.32,2.0,2.0,1000.0,0.0,0.0,0,False,False,...,False,False,False,False,False,False,False,True,False,False
3,0,1.60,3.0,4.0,1615.0,1.0,1.0,2,False,False,...,False,False,False,True,False,False,False,False,False,False
4,0,0.48,2.0,2.0,582.0,0.0,0.0,2,False,False,...,False,False,False,False,False,False,False,False,False,False


In [12]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [13]:
y_log = np.log1p(y)

In [14]:
y_log

0       0.598837
1       0.667829
2       0.277632
3       0.955511
4       0.392042
          ...   
3549    0.314811
3550    1.945910
3551    0.470004
3552    2.803360
3553    1.022451
Name: price, Length: 3554, dtype: float64

In [15]:
X

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna road,agePossession_old,agePossession_under construction
0,0,3.0,2.0,850.0,0.0,0.0,0,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,2.0,2.0,1226.0,1.0,0.0,0,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,2.0,2.0,1000.0,0.0,0.0,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,0,3.0,4.0,1615.0,1.0,1.0,2,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2.0,2.0,582.0,0.0,0.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,0,2.0,2.0,532.0,0.0,0.0,1,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3550,1,5.0,5.0,6228.0,1.0,0.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3551,0,1.0,1.0,665.0,0.0,1.0,1,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3552,1,5.0,6.0,5490.0,1.0,0.0,1,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [16]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [17]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [18]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna road,agePossession_old,agePossession_under construction
0,-0.517180,-0.074329,-0.874300,-0.831662,-0.747968,-0.668281,-0.984642,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
1,-0.517180,-0.877269,-0.874300,-0.522517,1.336956,-0.668281,-0.984642,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
2,-0.517180,-0.877269,-0.874300,-0.708333,-0.747968,-0.668281,-0.984642,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,4.561105,-0.602271,-0.290738
3,-0.517180,-0.074329,0.505173,-0.202684,1.336956,1.037949,1.866207,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,5.877074,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
4,-0.517180,-0.877269,-0.874300,-1.052010,-0.747968,-0.668281,1.866207,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,-0.517180,-0.877269,-0.874300,-1.093119,-0.747968,-0.668281,0.440783,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
3550,1.933563,1.531549,1.194909,3.590095,1.336956,-0.668281,1.866207,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,-0.602271,-0.290738
3551,-0.517180,-1.680208,-1.564036,-0.983768,-0.747968,1.037949,0.440783,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,1.660383,-0.290738
3552,1.933563,1.531549,1.884645,2.983317,1.336956,-0.668281,0.440783,-0.071348,-0.093805,-0.041123,...,-0.078923,-0.160267,-0.069328,-0.170153,-0.050386,-0.125375,-0.109357,-0.219245,1.660383,-0.290738


In [19]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [20]:
scores.mean(),scores.std()

(np.float64(0.8512613057405425), np.float64(0.016992929105286228))

In [21]:
lr = LinearRegression()


In [23]:
lr.fit(X_scaled, y_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](112,)","[ 0.12, 0.05, 0.07,...,-0.03,-0.01, 0.02]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](112,)","['property_type','bedRoom','bathroom',...,'sector_sohna road', 'agePossession_old','agePossession_under construction']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.046
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,112
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(112)


In [24]:
lr.coef_

array([ 1.20165036e-01,  5.40015778e-02,  6.51193988e-02,  2.10637561e-01,
        5.09461847e-02,  8.28171481e-03,  6.18569187e-03,  9.81917509e-03,
       -2.27136751e-02, -5.19263751e-03,  5.24027544e-03,  2.69999569e-02,
       -2.90628642e-03,  1.96385005e-03, -1.93922433e-02,  5.72258238e-04,
       -1.29872300e-02,  1.68170547e-02,  2.61496408e-02, -1.49660399e-02,
        8.73379008e-03,  1.62286944e-02,  3.08257676e-02,  3.21709327e-02,
       -1.94575986e-02, -1.22503903e-02,  2.87410970e-02,  3.29139011e-03,
        1.39170889e-02,  6.72176140e-03, -9.41789321e-03,  3.54737020e-02,
        2.19599278e-03,  1.77019789e-02,  5.75604298e-02,  7.38834785e-02,
        6.74158385e-03,  4.18540805e-02, -1.27147058e-02,  6.27615381e-03,
        2.24917111e-02,  2.59569793e-02,  2.49451572e-03, -1.14658162e-02,
        1.14313378e-03,  1.21374481e-02,  2.57701788e-03, -2.07745570e-02,
        7.65349283e-03,  1.67856368e-02,  6.03752164e-02,  2.69570318e-02,
        1.58617969e-02,  

In [25]:
lr.coef_.shape

(112,)

In [31]:
coef_df = pd.DataFrame(lr.coef_.reshape(1,112),columns=X.columns).stack().reset_index().drop(columns='level_0').rename(columns={'level_1':'Feature',0:'Coef'})
coef_df

,Feature,Coef
0,property_type,0.120165
1,bedRoom,0.054002
2,bathroom,0.065119
3,built_up_area,0.210638
4,servant room,0.050946
...,...,...
107,sector_sector 95,-0.025222
108,sector_sector 99,-0.010312
109,sector_sohna road,-0.029515
110,agePossession_old,-0.007900


In [32]:
y_log.std()

np.float64(0.5579613263072812)

In [33]:
X['bedRoom'].std()

np.float64(1.2455995038118572)

In [42]:
0.054 * (y_log.std()/X['bedRoom'].std())


np.float64(0.02418908447569853)

In [46]:
np.expm1(0.054 * (y_log.std()/X['bedRoom'].std()))

np.float64(0.024484013600191708)

In [48]:
np.expm1( 0.210638  * (y_log.std()/X['built_up_area'].std()))

np.float64(9.662172747465983e-05)

In [ ]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,112),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [675]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [628]:
lr.fit(X_scaled,y_log)

LinearRegression()

In [676]:
ridge.fit(X_scaled,y_log)

Ridge(alpha=0.0001)

In [677]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,112),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [678]:
coef_df

,feature,coef
0,property_type,0.120165
1,bedRoom,0.054002
2,bathroom,0.065119
3,built_up_area,0.210638
4,servant room,0.050946
...,...,...
107,sector_sector 95,-0.025222
108,sector_sector 99,-0.010312
109,sector_sohna road,-0.029515
110,agePossession_old,-0.007900


In [640]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     196.7
Date:                Tue, 26 Sep 2023   Prob (F-statistic):               0.00
Time:                        20:35:17   Log-Likelihood:                 588.22
No. Observations:                3554   AIC:                            -950.4
Df Residuals:                    3441   BIC:                            -252.6
Df Model:                         112                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [641]:
y_log.std()

0.5579613263072782

In [682]:
X_scaled['bedRoom'].std()

1.0001407162464122

In [683]:
0.21 * (0.557/1)

0.11697

In [681]:
np.expm1(0.030)

0.030454533953516858

In [653]:
2.4726962617564903e-05 * 100

0.0024726962617564905